# Notebook 01 — Exploratory Data Analysis

**Paper:** Predicting the Outcome of Transboundary Water Conflicts  
**Target journal:** *Global Environmental Change*  

This notebook covers:
1. Dataset overview and target distribution (Fig 1a — BAR_Scale histogram)
2. Temporal dynamics of conflict / cooperation (Fig 1b — stacked area, Fig 1c — events per year)
3. Data quality and missingness by feature group (Fig 1d)
4. Geographic distribution — top basins and continents (Fig 1e)
5. Feature correlations among retained predictors (Fig 1f)
6. Key bivariate relationships with BAR_Scale

**Dataset:** 6,805 events × 104 columns, years 1948–2008, 145 international river basins.  
**Target:** 4-class ordinal (0 = Conflict, 1 = Neutral, 2 = Mild Cooperation, 3 = Strong Cooperation).

## Cell 1 — Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ    = Path('..').resolve()
FIG_DIR = PROJ / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Class colour palette ───────────────────────────────────────────────────────
CLASS_COLORS = {
    0: '#c0392b',   # Conflict      — deep red
    1: '#e67e22',   # Neutral       — amber
    2: '#7fb3d3',   # Mild coop.    — light steel blue
    3: '#1a5276',   # Strong coop.  — dark navy blue
}
CLASS_LABELS = {
    0: 'Conflict',
    1: 'Neutral',
    2: 'Mild Cooperation',
    3: 'Strong Cooperation',
}

# ── Publication plot helpers ───────────────────────────────────────────────────
def apply_pub_style(fig, title='', xaxis_title='', yaxis_title='',
                    width=800, height=480, legend=True):
    """Apply Nature/Science publication styling to a Plotly figure."""
    fig.update_layout(
        font=dict(family='Arial', size=12, color='black'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        title=dict(text=title, font=dict(size=13, color='black')),
        xaxis=dict(
            showgrid=False, linecolor='black', linewidth=1,
            mirror=True, ticks='outside', title=xaxis_title,
            title_font=dict(size=12),
        ),
        yaxis=dict(
            showgrid=True, gridcolor='#e8e8e8', linecolor='black',
            linewidth=1, mirror=True, ticks='outside', title=yaxis_title,
            title_font=dict(size=12),
        ),
        margin=dict(l=65, r=25, t=50, b=65),
        width=width, height=height,
        showlegend=legend,
        legend=dict(
            bgcolor='white', bordercolor='black', borderwidth=0.5,
            font=dict(size=11),
        ),
    )
    return fig


def save_fig(fig, stem):
    """Save figure as HTML and high-resolution PNG (scale=2)."""
    html_path = FIG_DIR / f'{stem}.html'
    png_path  = FIG_DIR / f'{stem}.png'
    fig.write_html(str(html_path))
    fig.write_image(str(png_path), scale=2)
    print(f'  Saved: {html_path.name}  |  {png_path.name}')


print('Setup complete.')

Setup complete.


## Cell 2 — Load and inspect data

In [2]:
df = pd.read_parquet(PROJ / 'data' / 'processed' / 'events_enriched.parquet')

print(f"Shape:               {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Years:               {int(df['year'].min())}–{int(df['year'].max())}")
print(f"Unique basins:       {df['BCode'].nunique()}")
print(f"Events with BAR:     {df['BAR_Scale'].notna().sum():,}")
print()
print("Target distribution:")
tgt = df['target'].value_counts().sort_index()
for k, v in tgt.items():
    print(f"  {k} = {CLASS_LABELS[k]:<22} {v:5d}  ({v/len(df)*100:.1f}%)")
print()
print("BAR_Scale summary:")
print(df['BAR_Scale'].describe().round(3).to_string())

Shape:               6,805 rows × 104 columns
Years:               1948–2008
Unique basins:       145
Events with BAR:     6,805

Target distribution:
  0 = Conflict                1301  (19.1%)
  1 = Neutral                  273  (4.0%)
  2 = Mild Cooperation        3534  (51.9%)
  3 = Strong Cooperation      1697  (24.9%)

BAR_Scale summary:
count    6805.000
mean        1.682
std         2.316
min        -7.000
25%         1.000
50%         1.000
75%         3.000
max         6.000


## Cell 3 — Figure 1a: BAR_Scale distribution with 4-class grouping overlay

In [3]:
# Class boundary definitions
# conflict: BAR <= -1  | neutral: BAR == 0  | mild_coop: 1 <= BAR <= 3  | strong_coop: BAR >= 4
BAR_BOUNDS = [
    # (x_min, x_max, class_id)
    (-7.5, -0.5,  0),
    (-0.5,  0.5,  1),
    ( 0.5,  3.5,  2),
    ( 3.5,  6.5,  3),
]

bar_valid = df['BAR_Scale'].dropna()
n_total   = len(bar_valid)

fig1a = go.Figure()

# 1. Shaded class-region backgrounds (added before histogram so they sit behind)
alpha_band = 'rgba(%d,%d,%d,0.12)'
band_rgba = {
    0: 'rgba(192,57,43,0.12)',
    1: 'rgba(230,126,34,0.12)',
    2: 'rgba(127,179,211,0.12)',
    3: 'rgba(26,82,118,0.12)',
}
for x0, x1, cls in BAR_BOUNDS:
    fig1a.add_vrect(
        x0=x0, x1=x1,
        fillcolor=band_rgba[cls],
        layer='below', line_width=0,
    )

# 2. Histogram — all bars the same neutral grey, outlined
fig1a.add_trace(go.Histogram(
    x=bar_valid,
    nbinsx=28,
    marker=dict(color='#546e7a', line=dict(color='white', width=0.6)),
    opacity=0.90,
    name='Events',
    showlegend=False,
))

# 3. Vertical boundary lines
for xv, label in [(-0.5, ''), (0.5, ''), (3.5, '')]:
    fig1a.add_vline(
        x=xv, line=dict(color='#333333', width=1.5, dash='dash'),
    )

# 4. Class label annotations (above the plot area)
class_centers = [-3.5, 0.0, 2.0, 5.0]
for cls_id, (cx, (x0, x1, _)) in enumerate(zip(class_centers, BAR_BOUNDS)):
    n_cls = ((bar_valid >= x0) & (bar_valid < x1)).sum()
    pct   = n_cls / n_total * 100
    label = CLASS_LABELS[cls_id]
    fig1a.add_annotation(
        x=cx, y=1.04,
        xref='x', yref='paper',
        text=f'<b>{label}</b><br>{pct:.1f}%',
        showarrow=False,
        font=dict(size=10, color=list(CLASS_COLORS.values())[cls_id]),
        align='center',
    )

apply_pub_style(
    fig1a,
    title='Fig. 1a  —  BAR Scale distribution (n = 6,805)',
    xaxis_title='BAR Scale (−7 = most conflictual, +7 = most cooperative)',
    yaxis_title='Number of events',
    width=820, height=480, legend=False,
)
fig1a.update_xaxes(range=[-7.8, 7.0], dtick=1)
fig1a.show()
save_fig(fig1a, 'fig01a_bar_scale_distribution')

  Saved: fig01a_bar_scale_distribution.html  |  fig01a_bar_scale_distribution.png


## Cell 4 — Figure 1b: Temporal stacked area chart

In [4]:
# ── Build yearly proportions ───────────────────────────────────────────────────
yearly_counts = (
    df.groupby(['year', 'target'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=[0, 1, 2, 3], fill_value=0)
)
yearly_total = yearly_counts.sum(axis=1).replace(0, np.nan)
yearly_prop  = yearly_counts.div(yearly_total, axis=0).fillna(0)
years        = yearly_prop.index.astype(int).tolist()

# ── Smooth the cooperation ratio with a 5-year rolling mean ──────────────────
coop_ratio = (yearly_prop[2] + yearly_prop[3]).rolling(5, center=True, min_periods=3).mean()

fig1b = go.Figure()

# Stacked areas — ordered bottom to top: strong_coop, mild_coop, neutral, conflict
stack_order = [3, 2, 1, 0]   # bottom first
for cls in stack_order:
    fig1b.add_trace(go.Scatter(
        x=years,
        y=yearly_prop[cls].values,
        mode='lines',
        stackgroup='one',
        name=CLASS_LABELS[cls],
        line=dict(width=0.5, color=CLASS_COLORS[cls]),
        fillcolor=CLASS_COLORS[cls].replace('#', 'rgba(').rstrip(')') if False else CLASS_COLORS[cls],
        fill='tonexty',
        opacity=0.85,
        hovertemplate=f'{CLASS_LABELS[cls]}: %{{y:.1%}}<br>Year: %{{x}}<extra></extra>',
    ))

# Smoothed cooperation-ratio overlay
fig1b.add_trace(go.Scatter(
    x=years,
    y=coop_ratio.values,
    mode='lines',
    name='Coop. trend (5-yr MA)',
    line=dict(color='black', width=2, dash='dot'),
    hovertemplate='Coop. fraction (smoothed): %{y:.1%}<br>Year: %{x}<extra></extra>',
))

apply_pub_style(
    fig1b,
    title='Fig. 1b  —  Annual composition of water interaction outcomes (1948–2008)',
    xaxis_title='Year',
    yaxis_title='Proportion of events',
    width=900, height=460,
)
fig1b.update_yaxes(tickformat='.0%', range=[0, 1])
fig1b.update_xaxes(range=[1948, 2008], dtick=10)
fig1b.update_layout(
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='right', x=1,
        font=dict(size=10),
    )
)
fig1b.show()
save_fig(fig1b, 'fig01b_temporal_stacked')

  Saved: fig01b_temporal_stacked.html  |  fig01b_temporal_stacked.png


## Cell 5 — Figure 1c: Events per year with geopolitical annotations

In [5]:
events_per_year = df.groupby('year').size().reset_index(name='count')
events_per_year['year'] = events_per_year['year'].astype(int)

# 5-year rolling mean
events_per_year['smooth'] = events_per_year['count'].rolling(5, center=True, min_periods=3).mean()

fig1c = go.Figure()

# Raw bars (light)
fig1c.add_trace(go.Bar(
    x=events_per_year['year'],
    y=events_per_year['count'],
    name='Annual events',
    marker=dict(color='#7fb3d3', line=dict(color='#5a8fa8', width=0.4)),
    opacity=0.7,
))

# Smoothed line
fig1c.add_trace(go.Scatter(
    x=events_per_year['year'],
    y=events_per_year['smooth'],
    mode='lines',
    name='5-yr rolling mean',
    line=dict(color='#1a5276', width=2.5),
))

# Geopolitical milestones
milestones = [
    (1956, 'Suez Crisis', 'top'),
    (1967, '6-Day War', 'top'),
    (1978, 'Camp David', 'bottom'),
    (1991, 'Cold War end', 'top'),
    (1997, 'UN Watercourses\nConvention', 'bottom'),
]
for yr, label, position in milestones:
    ymax = events_per_year.loc[events_per_year['year'] == yr, 'count']
    yval = float(ymax.values[0]) if len(ymax) else 0
    ay   = -40 if position == 'top' else 40
    fig1c.add_annotation(
        x=yr, y=yval,
        text=label,
        showarrow=True,
        arrowhead=2, arrowsize=0.8, arrowwidth=1.2,
        arrowcolor='#555555',
        ay=ay, ax=0,
        font=dict(size=9, color='#333333'),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#aaaaaa', borderwidth=0.8,
    )
    fig1c.add_vline(
        x=yr, line=dict(color='#bbbbbb', width=1, dash='dot'),
        layer='below',
    )

apply_pub_style(
    fig1c,
    title='Fig. 1c  —  Water interaction events per year (1948–2008)',
    xaxis_title='Year',
    yaxis_title='Number of events',
    width=900, height=430,
)
fig1c.update_xaxes(range=[1947, 2009], dtick=5)
fig1c.show()
save_fig(fig1c, 'fig01c_events_per_year')

  Saved: fig01c_events_per_year.html  |  fig01c_events_per_year.png


## Cell 6 — Data quality / missingness report

In [6]:
# ── Define feature groups ──────────────────────────────────────────────────────
feature_groups = {
    'Basin characteristics (side 1)': [
        'Area_km2_1','PopDen2022_1','Dams_Exist_1','Dam_Plnd_1','EstDam24_1',
        'runoff_1','withdrawal_1','consumpt_1','HydroPolTe_1','InstitVuln_1',
        'NumberRipa_1','Wetlands_k_1','Count_of_t_1','Count_of_R_1',
    ],
    'Basin characteristics (side 2)': [
        'Area_km2_2','PopDen2022_2','Dams_Exist_2','Dam_Plnd_2','EstDam24_2',
        'runoff_2','withdrawal_2','consumpt_2','HydroPolTe_2','InstitVuln_2',
        'NumberRipa_2','Wetlands_k_2','Count_of_t_2','Count_of_R_2',
    ],
    'Climate (CHELSA)': ['pre','pre_ltm','pre_anomaly','pet','spei'],
    'Economic (WDI)': [
        'NY.GDP.PCAP.CD_wdi1','SP.POP.TOTL_wdi1',
        'MS.MIL.XPND.GD.ZS_wdi1','ER.H2O.FWTL.ZS_wdi1','ER.H2O.INTR.PC_wdi1',
        'NY.GDP.PCAP.CD_wdi2','SP.POP.TOTL_wdi2',
        'MS.MIL.XPND.GD.ZS_wdi2','ER.H2O.FWTL.ZS_wdi2','ER.H2O.INTR.PC_wdi2',
    ],
    'Governance (WGI / Polity)': [
        'CC.EST_wgi1','GE.EST_wgi1','PV.EST_wgi1','RL.EST_wgi1',
        'CC.EST_wgi2','GE.EST_wgi2','PV.EST_wgi2','RL.EST_wgi2',
        'polity2_pol1','polity2_pol2',
    ],
    'Water resources (AQUASTAT)': [
        'aq_intr_pc_aq1','aq_fwtl_zs_aq1','aq_fwag_zs_aq1','aq_fwdm_zs_aq1','aq_fwin_zs_aq1',
        'aq_intr_pc_aq2','aq_fwtl_zs_aq2','aq_fwag_zs_aq2','aq_fwdm_zs_aq2','aq_fwin_zs_aq2',
    ],
    'Temporal / dynamic': [
        'events_prior_5yr','cooperation_momentum','treaty_rate_5yr',
        'event_escalation','treaties_before_event',
    ],
    'Asymmetry indices': [
        'pop_ratio','withdrawal_ratio','dam_ratio','instit_vuln_diff',
        'hydropol_max','gdp_ratio','polity_diff','water_stress_diff',
    ],
}

# ── Compute mean non-null % per group ─────────────────────────────────────────
group_stats = []
for grp, cols in feature_groups.items():
    valid = [c for c in cols if c in df.columns]
    if not valid:
        continue
    pct_nonnull = df[valid].notna().mean().mean() * 100
    n_cols      = len(valid)
    group_stats.append({'group': grp, 'pct_nonnull': pct_nonnull, 'n_cols': n_cols})

gdf = pd.DataFrame(group_stats).sort_values('pct_nonnull')

# ── Horizontal bar chart ──────────────────────────────────────────────────────
bar_colors = [
    '#c0392b' if p < 60 else ('#e67e22' if p < 80 else '#27ae60')
    for p in gdf['pct_nonnull']
]

fig1d = go.Figure()
fig1d.add_trace(go.Bar(
    y=gdf['group'],
    x=gdf['pct_nonnull'],
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color='white', width=0.5)),
    text=[f"{p:.0f}%  (n={n})" for p, n in zip(gdf['pct_nonnull'], gdf['n_cols'])],
    textposition='outside',
    textfont=dict(size=10),
    showlegend=False,
))

# Reference lines
for xv, lbl, clr in [(60, '60%', '#c0392b'), (80, '80%', '#e67e22'), (100, '100%', '#555')]:
    fig1d.add_vline(
        x=xv, line=dict(color=clr, width=1, dash='dot'),
    )

apply_pub_style(
    fig1d,
    title='Fig. 1d  —  Data completeness by feature group',
    xaxis_title='Mean % non-null values',
    yaxis_title='',
    width=820, height=460, legend=False,
)
fig1d.update_xaxes(range=[0, 115], dtick=20)
fig1d.update_yaxes(tickfont=dict(size=11))
fig1d.show()
save_fig(fig1d, 'fig01d_missingness')

  Saved: fig01d_missingness.html  |  fig01d_missingness.png


## Cell 7 — Figure 1e: Top 10 basins by event count

In [7]:
# ── Top 10 basins by total event count ────────────────────────────────────────
top_basins_total = df['Basin_Name_1'].value_counts().head(10).index.tolist()

basin_class = (
    df[df['Basin_Name_1'].isin(top_basins_total)]
    .groupby(['Basin_Name_1', 'target'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2, 3], fill_value=0)
)
basin_class['total'] = basin_class.sum(axis=1)
basin_class['mean_bar'] = (
    df[df['Basin_Name_1'].isin(top_basins_total)]
    .groupby('Basin_Name_1')['BAR_Scale'].mean()
)
basin_class = basin_class.sort_values('total')

# ── Stacked horizontal bar chart, coloured by class ───────────────────────────
fig1e = go.Figure()
for cls in [0, 1, 2, 3]:
    fig1e.add_trace(go.Bar(
        y=basin_class.index,
        x=basin_class[cls],
        name=CLASS_LABELS[cls],
        orientation='h',
        marker=dict(color=CLASS_COLORS[cls], line=dict(color='white', width=0.4)),
        hovertemplate=f'{CLASS_LABELS[cls]}: %{{x}}<br>Basin: %{{y}}<extra></extra>',
    ))

# Mean BAR_Scale annotation
for basin in basin_class.index:
    total  = basin_class.loc[basin, 'total']
    mbar   = basin_class.loc[basin, 'mean_bar']
    fig1e.add_annotation(
        x=total + 12, y=basin,
        text=f'<i>μ={mbar:.1f}</i>',
        showarrow=False,
        font=dict(size=9, color='#444444'),
        xanchor='left',
    )

apply_pub_style(
    fig1e,
    title='Fig. 1e  —  Top 10 basins by event count (stacked by outcome class)',
    xaxis_title='Number of events',
    yaxis_title='',
    width=860, height=470,
)
fig1e.update_layout(
    barmode='stack',
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1, font=dict(size=10)),
)
fig1e.update_xaxes(range=[0, 900])
fig1e.update_yaxes(tickfont=dict(size=11))
fig1e.show()
save_fig(fig1e, 'fig01e_top_basins')

  Saved: fig01e_top_basins.html  |  fig01e_top_basins.png


## Cell 8 — Figure 1f: Feature correlation heatmap

In [8]:
# ── Select top-20 features by variance for readability ───────────────────────
EXCLUDE = {'ID', 'target', 'year', 'month', 'decade', 'cold_war', 'bilateral',
           'COPDAB_SCALE', 'BAR_Scale', 'Issue_Type1', 'Issue_Type2',
           'NUMBER_OF_BASINS', 'NUMBER_OF_Countries'}
num_feats = [
    c for c in df.select_dtypes('number').columns
    if c not in EXCLUDE
]

# Variance-based top-20 (sensible proxy for information content)
variances = df[num_feats].var().sort_values(ascending=False)

# Prefer a diverse set: grab top features but deduplicate near-mirror cols (_1/_2)
selected = []
seen_stems = set()
for col in variances.index:
    stem = col.rstrip('12').rstrip('_')
    # Allow both sides if informative but cap at 2 each
    count_stem = sum(1 for s in selected if s.rstrip('12').rstrip('_') == stem)
    if count_stem < 2:
        selected.append(col)
    if len(selected) >= 22:
        break

# Always include key predictors
must_include = [
    'treaties_before_event', 'cooperation_momentum', 'events_prior_5yr',
    'gdp_ratio', 'polity_diff', 'instit_vuln_diff', 'hydropol_max',
]
for c in must_include:
    if c in num_feats and c not in selected:
        selected.append(c)

selected = selected[:24]   # keep at most 24 for visual clarity

# Readable short labels
label_map = {
    'SP.POP.TOTL_wdi1': 'Population (S1)',
    'SP.POP.TOTL_wdi2': 'Population (S2)',
    'Area_km2_2': 'Basin area (S2)',
    'Area_km2_1': 'Basin area (S1)',
    'withdrawal_2': 'Water withdrawal (S2)',
    'withdrawal_1': 'Water withdrawal (S1)',
    'consumpt_2': 'Consumption (S2)',
    'consumpt_1': 'Consumption (S1)',
    'ER.H2O.INTR.PC_wdi1': 'Internal freshwater (S1)',
    'ER.H2O.INTR.PC_wdi2': 'Internal freshwater (S2)',
    'aq_intr_pc_aq1': 'Freshwater/cap AQUA (S1)',
    'aq_intr_pc_aq2': 'Freshwater/cap AQUA (S2)',
    'Wetlands_k_2': 'Wetlands area (S2)',
    'Wetlands_k_1': 'Wetlands area (S1)',
    'NY.GDP.PCAP.CD_wdi2': 'GDP/cap (S2)',
    'NY.GDP.PCAP.CD_wdi1': 'GDP/cap (S1)',
    'water_stress_diff': 'Water stress diff.',
    'aq_fwtl_zs_aq1': 'Freshwater total % (S1)',
    'ER.H2O.FWTL.ZS_wdi1': 'Freshwater total % WDI (S1)',
    'ER.H2O.FWTL.ZS_wdi2': 'Freshwater total % WDI (S2)',
    'treaties_before_event': 'Treaties (cumulative)',
    'cooperation_momentum': 'Coop. momentum',
    'events_prior_5yr': 'Events prior 5yr',
    'gdp_ratio': 'GDP ratio (S1/S2)',
    'polity_diff': 'Polity difference',
    'instit_vuln_diff': 'Inst. vuln. diff.',
    'hydropol_max': 'Hydropol. index (max)',
}

corr_df = df[selected].corr(method='spearman')
labels  = [label_map.get(c, c) for c in selected]

# Mask upper triangle
mask      = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
corr_vals = corr_df.values.copy()
corr_vals[mask] = np.nan

fig1f = go.Figure(go.Heatmap(
    z=corr_vals,
    x=labels,
    y=labels,
    colorscale='RdBu_r',
    zmid=0, zmin=-1, zmax=1,
    colorbar=dict(
        title=dict(text="Spearman ρ", side='right'),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        len=0.8,
    ),
    hovertemplate='%{y} × %{x}<br>ρ = %{z:.2f}<extra></extra>',
))

apply_pub_style(
    fig1f,
    title='Fig. 1f  —  Spearman correlation heatmap (top predictors)',
    xaxis_title='',
    yaxis_title='',
    width=880, height=780, legend=False,
)
fig1f.update_xaxes(tickangle=-40, tickfont=dict(size=9))
fig1f.update_yaxes(tickfont=dict(size=9), autorange='reversed')
fig1f.show()
save_fig(fig1f, 'fig01f_correlations')

  Saved: fig01f_correlations.html  |  fig01f_correlations.png


## Cell 9 — Bivariate relationships with BAR_Scale

In [9]:
biv_subset = df[[
    'treaties_before_event', 'cooperation_momentum',
    'NY.GDP.PCAP.CD_wdi1', 'NUMBER_OF_Countries',
    'BAR_Scale', 'target',
]].dropna(subset=['BAR_Scale'])

subplot_specs = [
    {
        'x_col': 'treaties_before_event',
        'x_label': 'Cumulative treaties in basin',
        'log_x': False,
        'title': 'Treaties vs. BAR Scale',
    },
    {
        'x_col': 'cooperation_momentum',
        'x_label': 'Cooperation momentum (5-yr avg)',
        'log_x': False,
        'title': 'Cooperation momentum vs. BAR Scale',
    },
    {
        'x_col': 'NY.GDP.PCAP.CD_wdi1',
        'x_label': 'GDP per capita, side 1 (USD, log scale)',
        'log_x': True,
        'title': 'GDP per capita (log) vs. BAR Scale',
    },
    {
        'x_col': 'NUMBER_OF_Countries',
        'x_label': 'Number of riparian countries',
        'log_x': False,
        'title': 'Riparian country count vs. BAR Scale',
    },
]

fig_biv = make_subplots(
    rows=2, cols=2,
    horizontal_spacing=0.12,
    vertical_spacing=0.15,
    subplot_titles=[sp['title'] for sp in subplot_specs],
)

positions = [(1,1),(1,2),(2,1),(2,2)]

for (row, col), sp in zip(positions, subplot_specs):
    sub = biv_subset[[sp['x_col'], 'BAR_Scale', 'target']].dropna()
    x_vals = sub[sp['x_col']]
    if sp['log_x']:
        x_plot = np.log10(x_vals.clip(lower=1))
    else:
        x_plot = x_vals

    # Scatter by class
    for cls in [0, 1, 2, 3]:
        mask = sub['target'] == cls
        fig_biv.add_trace(
            go.Scatter(
                x=x_plot[mask],
                y=sub.loc[mask, 'BAR_Scale'],
                mode='markers',
                marker=dict(
                    color=CLASS_COLORS[cls],
                    size=4, opacity=0.45,
                    line=dict(width=0),
                ),
                name=CLASS_LABELS[cls],
                legendgroup=str(cls),
                showlegend=(row == 1 and col == 1),
                hovertemplate=(
                    f"{sp['x_label']}: %{{x:.2f}}<br>"
                    f"BAR Scale: %{{y}}<extra>{CLASS_LABELS[cls]}</extra>"
                ),
            ),
            row=row, col=col,
        )

    # OLS trend line
    x_arr = x_plot.values
    y_arr = sub['BAR_Scale'].values
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    if finite.sum() > 10:
        p = np.polyfit(x_arr[finite], y_arr[finite], 1)
        x_line = np.linspace(x_arr[finite].min(), x_arr[finite].max(), 200)
        y_line = np.polyval(p, x_line)
        fig_biv.add_trace(
            go.Scatter(
                x=x_line, y=y_line,
                mode='lines',
                line=dict(color='black', width=1.8, dash='solid'),
                showlegend=False,
                hoverinfo='skip',
            ),
            row=row, col=col,
        )

    # Per-panel axis labels
    axis_idx = '' if (row == 1 and col == 1) else str((row-1)*2 + col)
    x_key = f'xaxis{axis_idx}' if axis_idx != '1' else 'xaxis'
    y_key = f'yaxis{axis_idx}' if axis_idx != '1' else 'yaxis'
    fig_biv.update_layout(**{
        x_key: dict(title=sp['x_label'], title_font=dict(size=10)),
        y_key: dict(title='BAR Scale', title_font=dict(size=10)),
    })

fig_biv.update_layout(
    font=dict(family='Arial', size=11, color='black'),
    plot_bgcolor='white', paper_bgcolor='white',
    width=900, height=680,
    margin=dict(l=60, r=20, t=60, b=60),
    title=dict(
        text='Bivariate relationships with BAR Scale by outcome class',
        font=dict(size=13),
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.12,
        xanchor='center', x=0.5, font=dict(size=10),
        bgcolor='white', bordercolor='black', borderwidth=0.5,
    ),
)
for axis in fig_biv.layout:
    if axis.startswith('xaxis'):
        fig_biv.layout[axis].update(
            showgrid=False, linecolor='black', linewidth=1,
            mirror=True, ticks='outside',
        )
    elif axis.startswith('yaxis'):
        fig_biv.layout[axis].update(
            showgrid=True, gridcolor='#e8e8e8', linecolor='black',
            linewidth=1, mirror=True, ticks='outside',
        )

fig_biv.show()
# Note: bivariate panel not in the fig01 save list — saved for supplementary
html_path = FIG_DIR / 'fig01_bivariate_relationships.html'
png_path  = FIG_DIR / 'fig01_bivariate_relationships.png'
fig_biv.write_html(str(html_path))
fig_biv.write_image(str(png_path), scale=2)
print(f'  Saved: {html_path.name}  |  {png_path.name}')

  Saved: fig01_bivariate_relationships.html  |  fig01_bivariate_relationships.png


## Cell 10 — Geographic distribution by continent and class

In [10]:
CONTINENT_NAMES = {
    'AS': 'Asia',
    'EU': 'Europe',
    'AF': 'Africa',
    'SA': 'South America',
    'NA': 'North America',
    'OC': 'Oceania',
}

cont_class = (
    df.groupby(['Continent', 'target'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=[0, 1, 2, 3], fill_value=0)
)
cont_class['total'] = cont_class.sum(axis=1)
cont_class = cont_class.sort_values('total', ascending=True)
cont_labels = [CONTINENT_NAMES.get(c, c) for c in cont_class.index]

fig_cont = go.Figure()
for cls in [0, 1, 2, 3]:
    fig_cont.add_trace(go.Bar(
        y=cont_labels,
        x=cont_class[cls],
        name=CLASS_LABELS[cls],
        orientation='h',
        marker=dict(color=CLASS_COLORS[cls], line=dict(color='white', width=0.5)),
        hovertemplate=f'{CLASS_LABELS[cls]}: %{{x}}<br>%{{y}}<extra></extra>',
    ))

apply_pub_style(
    fig_cont,
    title='Geographic distribution of water interaction events by outcome class',
    xaxis_title='Number of events',
    yaxis_title='',
    width=800, height=360,
)
fig_cont.update_layout(
    barmode='stack',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
)
fig_cont.show()
# Save as supplementary
html_path = FIG_DIR / 'fig01_continent_distribution.html'
png_path  = FIG_DIR / 'fig01_continent_distribution.png'
fig_cont.write_html(str(html_path))
fig_cont.write_image(str(png_path), scale=2)
print(f'  Saved: {html_path.name}  |  {png_path.name}')

  Saved: fig01_continent_distribution.html  |  fig01_continent_distribution.png


## Summary statistics table

In [11]:
summary_cols = [
    'BAR_Scale', 'treaties_before_event', 'cooperation_momentum',
    'events_prior_5yr', 'NY.GDP.PCAP.CD_wdi1', 'polity2_pol1',
    'HydroPolTe_1', 'InstitVuln_1', 'spei',
]
summary_labels = [
    'BAR Scale', 'Cumulative treaties', 'Cooperation momentum',
    'Events prior 5yr', 'GDP per capita (S1, USD)', 'Polity score (S1)',
    'Hydropolitical tension (S1)', 'Institutional vulnerability (S1)', 'SPEI drought index',
]
valid = [c for c in summary_cols if c in df.columns]

summary = df[valid].describe().T.round(2)
summary.index = [summary_labels[summary_cols.index(c)] for c in valid]
summary['missing %'] = (df[valid].isna().mean() * 100).round(1).values

print(summary[['count','mean','std','min','25%','50%','75%','max','missing %']].to_string())

                                   count     mean      std    min     25%     50%      75%       max  missing %
BAR Scale                         6805.0     1.68     2.32  -7.00    1.00    1.00     3.00      6.00        0.0
Cumulative treaties               6805.0    57.91    75.89   0.00   14.00   29.00    60.00    326.00        0.0
Cooperation momentum              6264.0     1.55     1.44  -7.00    0.67    1.40     2.23      6.00        8.0
Events prior 5yr                  6790.0    76.94    91.79   0.00    9.00   45.00   112.00    504.00        0.2
GDP per capita (S1, USD)          5640.0  4084.07  7634.53  17.33  394.66  992.52  2867.70  64989.16       17.1
Polity score (S1)                 4077.0     0.08     7.63 -10.00   -7.00   -2.00     8.00     10.00       40.1
Hydropolitical tension (S1)       5210.0     2.57     1.00   1.00    2.00    3.00     3.00      5.00       23.4
Institutional vulnerability (S1)  5210.0     4.19     1.35   0.00    4.00    5.00     5.00      5.00    